# 数独循环实验 · 受保护的残差通道

在相同隐藏宽度和训练样本下，检验分区写门能否减少空格答案的 **对→错**，同时保留 **错→对**。这份 notebook 只运行小规模试验。正式训练的九次运行以计时外推。

**Colab 操作**　选择 GPU，点击“全部运行”。在右上角资源面板读取当前运行时的 **CU/小时**，按提示输入。使用新运行时，保持单个活跃会话。如果为同一次试跑更换了运行时，先在最后一个代码单元的 `PREVIOUS_CU` 填入此前已用额度。输入框等待最多 3 分钟。

**预算**　授权上限 10 CU，程序按当前费率在 8 CU 或 2 小时的较早时点请求释放 GPU，并提前 3 分钟收尾。连接前后的计费差额、费率变更及服务中断以 Colab 资源面板为准。运行完成或发生错误时，结果写到本单元输出，随后自动释放运行时。下载 JSON 的链接嵌在结果下方。


In [1]:
import time
if 'SUDOKU_SESSION_START' not in globals():
    SUDOKU_SESSION_START = time.monotonic()


## 数据和对照

每题含 81 个位置，0 表示空格，标签为唯一终局。离线生成 2,048 道训练题、256 道验证题和 256 道封存测试题。按终局的对称不变量划分，同一终局的数字重命名及合法行列置换等变形保持在同一划分。试跑只查看验证集。训练的在线数字重命名、碎批顺序及主干初始化对三组配对。

同一个 Transformer 块循环 8 次。总宽 96，FFN 宽 192；四个注意力头分别看同行、同列、同宫和全盘，只依赖题面的位置关系。输出九类数字，仅对原始空格计算交叉熵。三个模型使用相同的最终轮监督。

| 模型 | 写入规则 |
|---|---|
| 普通循环 | 96 维都接受本轮候选状态。 |
| 全通道门控 | 32 个写门覆盖全部 96 维，三组通道共享同一组门值。 |
| 分区门控 | 同样的 32 个门只控制前 32 维；后 64 维自由更新。 |

门控对照有相同的参数量和门投影计算。普通组没有门参数。模型能读取所有通道；保护区没有绕过门的残差写入路径。


In [2]:
# 可离线生成的 9×9 数独。导出为 notebook 单元后只依赖 Python 和 NumPy。
import hashlib
import random
import numpy as np

_SDK_ALL = (1 << 9) - 1
_SDK_ROW = tuple(i // 9 for i in range(81))
_SDK_COL = tuple(i % 9 for i in range(81))
_SDK_BOX = tuple(i // 27 * 3 + i % 9 // 3 for i in range(81))


def _sdk_completion(puzzle, *, known=None, rng=None):
    """Return one completion, excluding known when supplied. Internal grids are valid."""
    board = list(puzzle)
    rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
    empty = []
    for i, digit in enumerate(board):
        if digit:
            bit = 1 << (digit - 1)
            rows[_SDK_ROW[i]] |= bit
            cols[_SDK_COL[i]] |= bit
            boxes[_SDK_BOX[i]] |= bit
        else:
            empty.append(i)
    if rng is not None:
        rng.shuffle(empty)

    def search(depth, differs):
        if depth == len(empty):
            return board.copy() if known is None or differs else None
        best, best_mask, best_n = depth, 0, 10
        for j in range(depth, len(empty)):
            i = empty[j]
            mask = _SDK_ALL & ~(rows[_SDK_ROW[i]] | cols[_SDK_COL[i]] | boxes[_SDK_BOX[i]])
            count = mask.bit_count()
            if count < best_n:
                if count == 0:
                    return None
                best, best_mask, best_n = j, mask, count
                if count == 1:
                    break
        empty[depth], empty[best] = empty[best], empty[depth]
        i = empty[depth]
        r, c, b = _SDK_ROW[i], _SDK_COL[i], _SDK_BOX[i]
        candidates = []
        while best_mask:
            bit = best_mask & -best_mask
            candidates.append(bit)
            best_mask ^= bit
        if best_n > 1:
            if rng is not None:
                rng.shuffle(candidates)
            elif known is not None and not differs:
                preferred_last = 1 << (known[i] - 1)
                if preferred_last in candidates:
                    candidates.remove(preferred_last)
                    candidates.append(preferred_last)
        for bit in candidates:
            digit = bit.bit_length()
            board[i] = digit
            rows[r] |= bit
            cols[c] |= bit
            boxes[b] |= bit
            result = search(depth + 1, differs or (known is not None and digit != known[i]))
            if result is not None:
                return result
            rows[r] ^= bit
            cols[c] ^= bit
            boxes[b] ^= bit
        board[i] = 0
        empty[depth], empty[best] = empty[best], empty[depth]
        return None

    return search(0, False)


def _sdk_split_id(solution):
    # The cycle lengths of the digit permutation formed by two rows survive
    # digit relabeling and column permutation. Sorting both orientations also
    # makes the signature invariant to row order and transposition. Thus every
    # board in a Sudoku symmetry orbit is assigned to the same split. Distinct
    # orbits can share a signature, which only keeps extra examples together.
    rows = [solution[i : i + 9] for i in range(0, 81, 9)]

    def fingerprint(lines):
        result = []
        for i, left in enumerate(lines):
            for right in lines[i + 1 :]:
                perm = [0] * 10
                for a, b in zip(left, right):
                    perm[a] = b
                seen, lengths = 0, []
                for digit in range(1, 10):
                    length = 0
                    while not (seen & (1 << digit)):
                        seen |= 1 << digit
                        digit = perm[digit]
                        length += 1
                    if length:
                        lengths.append(length)
                result.append(tuple(sorted(lengths)))
        return tuple(sorted(result))

    signature = sorted((fingerprint(rows), fingerprint(list(zip(*rows)))))
    digest = hashlib.blake2b(repr(signature).encode("ascii"), digest_size=8).digest()
    bucket = int.from_bytes(digest, "little") % 10
    return "val" if bucket == 8 else "test" if bucket == 9 else "train"


def _sdk_puzzle(solution, rng, index):
    # Mix starting clue densities. Clues added to force uniqueness count too.
    low, high = ((24, 31), (32, 40), (43, 52))[index % 3]
    puzzle = [0] * 81
    for i in rng.sample(range(81), rng.randint(low, high)):
        puzzle[i] = solution[i]
    while True:
        alternative = _sdk_completion(puzzle, known=solution)
        if alternative is None:
            return puzzle
        i = rng.choice([i for i in range(81) if alternative[i] != solution[i]])
        puzzle[i] = solution[i]


def make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026):
    """Return {train, val, test: (puzzles[N,81], solutions[N,81])}, uint8."""
    sizes = dict(train=n_train, val=n_val, test=n_test)
    if any(not isinstance(n, (int, np.integer)) or n < 0 for n in sizes.values()):
        raise ValueError("Split sizes must be nonnegative integers.")
    rng = random.Random(int(seed))
    splits = {k: (np.empty((n, 81), np.uint8), np.empty((n, 81), np.uint8))
              for k, n in sizes.items()}
    counts = dict.fromkeys(sizes, 0)
    seen = set()
    while counts != sizes:
        solution = _sdk_completion([0] * 81, rng=rng)
        puzzle_seed = rng.getrandbits(64)
        key = _sdk_split_id(solution)
        i = counts[key]
        code = bytes(solution)
        if i == sizes[key] or code in seen:
            continue
        seen.add(code)
        puzzle = _sdk_puzzle(solution, random.Random(puzzle_seed), i)
        splits[key][0][i] = puzzle
        splits[key][1][i] = solution
        counts[key] += 1
    return splits


def sample_sudoku_batch(split, batch_size, rng, augment=True):
    """Sample with replacement. Pass a seeded np.random.Generator. No split mutation."""
    puzzles, solutions = split
    picks = rng.integers(0, len(puzzles), size=batch_size)
    puzzles, solutions = puzzles[picks], solutions[picks]
    if not augment:
        return puzzles, solutions

    def shuffled_axis():
        return np.concatenate([3 * band + rng.permutation(3) for band in rng.permutation(3)])

    grid = np.arange(81).reshape(9, 9)
    if rng.integers(2):
        grid = grid.T
    positions = grid[np.ix_(shuffled_axis(), shuffled_axis())].reshape(-1)
    digit_map = np.zeros((batch_size, 10), np.uint8)
    digit_map[:, 1:] = 1 + np.argsort(rng.random((batch_size, 9)), axis=1)
    return tuple(np.take_along_axis(digit_map, values[:, positions], axis=1)
                 for values in (puzzles, solutions))


In [3]:
import torch
from torch import nn
from torch.nn import functional as F


class SudokuBlock(nn.Module):
    """One shared pre-norm Transformer block with bidirectional attention."""

    def __init__(self, width, heads, ffn_width):
        super().__init__()
        self.heads = heads
        self.head_width = width // heads
        self.attn_norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.attn_out = nn.Linear(width, width)
        positions = torch.arange(81)
        rows, cols = positions // 9, positions % 9
        boxes = (rows // 3) * 3 + cols // 3
        same_row = rows[:, None] == rows[None, :]
        same_col = cols[:, None] == cols[None, :]
        same_box = boxes[:, None] == boxes[None, :]
        global_attention = torch.ones_like(same_row)
        head_types = (same_row, same_col, same_box, global_attention)
        masks = [head_types[i % 4] for i in range(heads)]
        if heads < 4:
            masks[-1] = global_attention
        # SDPA True entries permit attention. Shared by all variants, this
        # puzzle-topology prior contains only locations, never puzzle answers.
        self.register_buffer("attention_mask", torch.stack(masks)[None], persistent=False)
        self.ffn_norm = nn.LayerNorm(width)
        self.ffn = nn.Sequential(
            nn.Linear(width, ffn_width), nn.GELU(), nn.Linear(ffn_width, width)
        )

    def forward(self, state):
        batch, cells, width = state.shape
        qkv = self.qkv(self.attn_norm(state))
        qkv = qkv.reshape(batch, cells, 3, self.heads, self.head_width)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        context = F.scaled_dot_product_attention(
            query, key, value, attn_mask=self.attention_mask, is_causal=False
        )
        context = context.transpose(1, 2).reshape(batch, cells, width)
        proposal = state + self.attn_out(context)
        return proposal + self.ffn(self.ffn_norm(proposal))


class SudokuLoopTransformer(nn.Module):
    """Inputs are digits 0–9 (0 = blank). Logit indices 0–8 denote digits 1–9."""

    def __init__(self, variant, width=96, heads=4, ffn_width=192, core_width=32):
        super().__init__()
        if variant not in ("plain", "full_gate", "split_gate"):
            raise ValueError(f"Unknown model variant: {variant}")
        if heads < 1 or width % heads or not 0 < core_width < width:
            raise ValueError("Use a width divisible by heads and 0 < core_width < width.")
        self.variant = variant
        self.digit = nn.Embedding(10, width)
        self.row = nn.Embedding(9, width)
        self.col = nn.Embedding(9, width)
        self.box = nn.Embedding(9, width)
        cell = torch.arange(81)
        rows, cols = cell // 9, cell % 9
        self.register_buffer("rows", rows, persistent=False)
        self.register_buffer("cols", cols, persistent=False)
        self.register_buffer("boxes", (rows // 3) * 3 + cols // 3, persistent=False)
        self.block = SudokuBlock(width, heads, ffn_width)
        self.readout = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 9))
        if variant != "plain":
            self.gate = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, core_width))
            # A full-stream gate shares one decision per group of channels. Both
            # controls have the same number of learned gates and the same GEMMs.
            channel = torch.arange(width)
            self.register_buffer("gate_group", channel % core_width, persistent=False)
            gated = channel < core_width if variant == "split_gate" else channel >= 0
            self.register_buffer("gated_channels", gated, persistent=False)

    def initial_state(self, puzzles):
        xy = self.row(self.rows) + self.col(self.cols) + self.box(self.boxes)
        return 0.5 * (self.digit(puzzles.long()) + xy)

    def step(self, state, gate_mode="learned"):
        proposal = self.block(state)
        if self.variant == "plain":
            return proposal
        gate = torch.sigmoid(self.gate(proposal)).to(dtype=state.dtype)
        if gate_mode == "open":
            gate = torch.ones_like(gate)
        elif gate_mode == "closed":
            gate = torch.zeros_like(gate)
        elif gate_mode != "learned":
            raise ValueError(f"Unknown gate intervention: {gate_mode}")
        gate = gate.index_select(-1, self.gate_group)
        # Only this interpolation can write the protected channels. Attention
        # and FFN proposals can read all channels. The readout never writes back.
        gated_proposal = torch.lerp(state, proposal, gate)
        return torch.where(self.gated_channels, gated_proposal, proposal)

    def forward(self, puzzles, steps, return_all=False, gate_mode="learned"):
        if puzzles.ndim != 2 or puzzles.shape[1] != 81 or steps < 1:
            raise ValueError("Expected [batch, 81] puzzles and steps >= 1.")
        state = self.initial_state(puzzles)
        if return_all:
            logits = []
            for _ in range(steps):
                state = self.step(state, gate_mode=gate_mode)
                logits.append(self.readout(state))
            return torch.stack(logits, dim=1)  # [batch, steps, 81, 9]
        for _ in range(steps):
            state = self.step(state, gate_mode=gate_mode)
        return self.readout(state)  # [batch, 81, 9]


In [4]:
import math
import sys
import threading
import time


class BudgetStop(RuntimeError):
    """The experiment must stop and preserve its current results."""


def release_colab_runtime():
    """Release the hosted VM using Colab's public runtime-management function."""
    from google.colab import runtime

    try:
        runtime.unassign()
    except Exception:
        # Retry once in case the backend briefly lost the connection.
        time.sleep(2)
        runtime.unassign()


class RuntimeBudget:
    """Wall-clock guard using the rate shown by the active Colab runtime."""

    def __init__(self, cu_per_hour, already_used_cu=0.0, *, session_start,
                 limit_cu=8.0, max_hours=2.0, reserve_minutes=3.0):
        self.rate = float(cu_per_hour)
        self.prior_cu = float(already_used_cu)
        self.start = float(session_start)
        self.limit_cu = float(limit_cu)
        max_hours = float(max_hours)
        reserve_minutes = float(reserve_minutes)
        values = (self.rate, self.prior_cu, self.start, self.limit_cu,
                  max_hours, reserve_minutes)
        if not all(math.isfinite(value) for value in values):
            raise ValueError("CU 费率、预算和时间都必须是有限数值。")
        if self.rate <= 0 or self.prior_cu < 0:
            raise ValueError("请填写资源面板当前的正数 CU/小时和非负的已用 CU。")
        if not (0 < self.limit_cu <= 8 and 0 < max_hours <= 2 and reserve_minutes > 0):
            raise ValueError("本实验的运行预算上限为 8 CU、2 小时；收尾时间须为正数。")
        if not 0 <= self.start <= time.monotonic():
            raise ValueError("session_start 应取自当前内核首次运行时的 time.monotonic()。")
        available_cu = max(0.0, self.limit_cu - self.prior_cu)
        allowed_seconds = 3600 * min(max_hours, available_cu / self.rate)
        self.release_deadline = self.start + allowed_seconds
        self.work_deadline = self.release_deadline - 60 * reserve_minutes
        self._watchdog = None

    def remaining_seconds(self):
        """Seconds available for training or evaluation before wrap-up begins."""
        return max(0.0, self.work_deadline - time.monotonic())

    def check(self):
        if self.remaining_seconds() == 0:
            raise BudgetStop("已到预留的收尾时间，停止训练并保存当前结果。")

    def summary(self):
        elapsed = max(0.0, time.monotonic() - self.start)
        return {
            "runtime_cu_per_hour_entered": self.rate,
            "previous_sessions_cu_entered": self.prior_cu,
            "this_session_seconds": round(elapsed, 2),
            "estimated_total_cu": round(self.prior_cu + self.rate * elapsed / 3600, 4),
            "software_cu_limit": self.limit_cu,
            "remaining_work_seconds": round(self.remaining_seconds(), 2),
        }

    def arm_watchdog(self):
        """Request VM release at the ceiling even if a training call blocks."""
        if self._watchdog is not None:
            return

        def on_deadline():
            print("已到运行时限，正在释放 Colab 运行时。", file=sys.stderr, flush=True)
            try:
                release_colab_runtime()
            except Exception as exc:
                print(f"Colab 自动释放失败，请手动断开并删除运行时。{exc}",
                      file=sys.stderr, flush=True)

        seconds = max(0.0, self.release_deadline - time.monotonic())
        self._watchdog = threading.Timer(seconds, on_deadline)
        self._watchdog.daemon = True
        self._watchdog.start()

    def disarm_watchdog(self):
        if self._watchdog is not None:
            self._watchdog.cancel()
            self._watchdog = None


## 计时与评估

预热 16 个真实训练步，随后以 16 步为一段，用 GPU 同步后的墙钟秒数计时，涵盖取样、增强、前后向、裁剪梯度和优化器更新。三组轮换，每组最多 512 步。验证一次向前运行到 32 轮，在第 8、16、32 轮读取结果。整盘正确要求所有原空格填对，给定数字沿用题面。

对→错率的分母是第 8 轮答对的空格数，错→对率的分母是第 8 轮答错的空格数。观察两者并结合整盘正确率；一次单种子、短训练的结果用于判断任务是否产生足够可观察的翻转。

正式预算按三组各三种子、每种子 5,000 步计算。逐组使用自己的分段速度、预热和一次同规模验证的实测时间。报告同时给出较慢分段情景和额外留 20% 的预算，产生测试集结果、额外评估、频繁存盘、调参和空闲等待的费用需另计。


In [5]:
import json
import math
import time
from pathlib import Path
from contextlib import nullcontext
from html import escape
from urllib.parse import quote

import numpy as np
import torch
from IPython.display import HTML, display


VARIANTS = ('plain', 'full_gate', 'split_gate')
NAMES = {'plain': '普通循环', 'full_gate': '全通道门控', 'split_gate': '分区门控'}
CHECK_DEPTHS = (8, 16, 32)


def evaluate_loop(model, validation, device, precision, batch_size, budget):
    model.eval()
    correct_counts = {d: [0, 0] for d in CHECK_DEPTHS}
    flips = {d: [0, 0] for d in CHECK_DEPTHS[1:]}
    n_blank = 0
    with torch.inference_mode():
        for start in range(0, len(validation[0]), batch_size):
            budget.check()
            p, y = [torch.as_tensor(a[start:start + batch_size], device=device).long() for a in validation]
            mask = p.eq(0)
            n_blank += int(mask.sum())
            with precision():
                state = model.initial_state(p)
                for depth in range(1, max(CHECK_DEPTHS) + 1):
                    state = model.step(state)
                    if depth not in CHECK_DEPTHS:
                        continue
                    correct = model.readout(state).argmax(-1).add(1).eq(y)
                    correct_counts[depth][0] += int((correct & mask).sum())
                    correct_counts[depth][1] += int((correct | ~mask).all(-1).sum())
                    if depth == CHECK_DEPTHS[0]:
                        at_eight = correct
                    else:
                        flips[depth][0] += int((mask & at_eight & ~correct).sum())
                        flips[depth][1] += int((mask & ~at_eight & correct).sum())
    n_boards = len(validation[0])
    n_right = correct_counts[8][0]
    def ratio(n, d):
        return n / d if d else None
    metrics = []
    for depth, (cells, boards) in correct_counts.items():
        row = {'depth': depth, 'blank_acc': ratio(cells, n_blank), 'board_acc': ratio(boards, n_boards),
               'blanks': n_blank, 'boards': n_boards, 'correct_blanks': cells, 'solved_boards': boards}
        if depth in flips:
            ruined, repaired = flips[depth]
            assert cells == n_right - ruined + repaired
            row.update(right_to_wrong=ratio(ruined, n_right), wrong_to_right=ratio(repaired, n_blank-n_right),
                       ruined=ruined, initially_right=n_right, repaired=repaired, initially_wrong=n_blank-n_right)
        metrics.append(row)
    return metrics


def formal_forecast(arms, rate, planned_steps, seeds, timed_warmup):
    # The high scenario uses the 90th percentile of measured block means. It is not a CI.
    rows = {}
    for name, arm in arms.items():
        blocks = arm['timings']
        if not blocks:
            continue
        avg = sum(b['seconds'] for b in blocks) / sum(b['steps'] for b in blocks)
        slow = max(avg, float(np.quantile([b['seconds']/b['steps'] for b in blocks], 0.90)))
        # Per formal seed, use actual first-block time and one measured final 32-loop evaluation.
        remaining = max(0, planned_steps - timed_warmup)
        fixed = arm['warmup_seconds'] + arm.get('eval_seconds', 0.0)
        usual_seconds = seeds * (fixed + remaining * avg)
        slow_seconds = seeds * (fixed + remaining * slow)
        rows[name] = {'seconds_per_step': avg, 'timed_steps': sum(b['steps'] for b in blocks),
                      'timing_blocks': len(blocks), 'hours': usual_seconds/3600,
                      'cu': usual_seconds*rate/3600, 'slower_cu': slow_seconds*rate/3600}
    if len(rows) < len(VARIANTS):
        return {'by_model': rows, 'total': None, 'reason': '部分模型缺少预热后的计时，无法外推全部九次训练。'}
    cu = sum(r['cu'] for r in rows.values())
    return {'by_model': rows, 'total': {'hours': sum(r['hours'] for r in rows.values()), 'cu': cu,
            'slower_cu': sum(r['slower_cu'] for r in rows.values()), 'cu_plus_20pct': cu*1.2,
            'includes_one_validation_per_seed': all('eval_seconds' in a for a in arms.values())}}


def run_pilot(budget, rate, report, config, output_dir, device='cuda'):
    torch.set_float32_matmul_precision('high')
    cuda = torch.device(device).type == 'cuda'
    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if cuda else torch.float32
    precision = (lambda: torch.autocast(device_type='cuda', dtype=dtype)) if cuda else nullcontext
    sync = torch.cuda.synchronize if cuda else lambda: None
    report.update(config=dict(config), gpu=torch.cuda.get_device_name() if cuda else 'CPU',
                  torch_version=str(torch.__version__), precision=str(dtype), cu_per_hour=rate, arms={})
    print(f"设备 {report['gpu']}，{dtype}，试跑上限 10 CU。开始准备离线数独…", flush=True)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    splits = make_sudoku_splits(n_train=config['n_train'], n_val=config['n_val'], n_test=config['n_test'], seed=config['data_seed'])
    report['data_seconds'] = time.perf_counter() - started
    budget.check()
    print(f"数据已就绪，用时 {report['data_seconds']:.1f}s，验证题数 {len(splits['val'][0])}。", flush=True)
    train_x, train_y = [torch.as_tensor(a, device=device).long() for a in splits['train']]
    models, opts, scalers, generators = {}, {}, {}, {}
    for name in VARIANTS:
        torch.manual_seed(config['seed'])
        model = SudokuLoopTransformer(name, width=config['width'], heads=4, ffn_width=config['width']*2,
                                      core_width=config['width']//3).to(device)
        models[name] = model
        opts[name] = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
        scalers[name] = torch.amp.GradScaler('cuda', enabled=cuda and dtype == torch.float16, init_scale=256.0)
        generators[name] = torch.Generator(device=device).manual_seed(config['batch_seed'])
        report['arms'][name] = dict(steps=0, warmup_seconds=0.0, timings=[], params=sum(p.numel() for p in model.parameters()))

    def block(name, count, warmup=False):
        model, opt, scaler, gen = models[name], opts[name], scalers[name], generators[name]
        model.train()
        sync()
        before = time.perf_counter()
        for _ in range(count):
            budget.check()
            ids = torch.randint(len(train_x), (config['batch'],), device=device, generator=gen)
            x, y = train_x[ids], train_y[ids]
            # Uniform digit relabeling per board; zero remains the blank symbol.
            perm = torch.rand((len(x), 9), device=device, generator=gen).argsort(1) + 1
            perm = torch.cat((torch.zeros_like(perm[:, :1]), perm), dim=1)
            x, y = perm.gather(1, x), perm.gather(1, y)
            target = (y - 1).masked_fill(x.ne(0), -100)
            opt.zero_grad(set_to_none=True)
            with precision():
                logits = model(x, steps=config['train_loops'])
                loss = torch.nn.functional.cross_entropy(logits.reshape(-1, 9), target.reshape(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            report['arms'][name]['steps'] += 1
        sync()
        elapsed = time.perf_counter() - before
        arm = report['arms'][name]
        if warmup:
            arm['warmup_seconds'] = elapsed
        else:
            arm['timings'].append({'seconds': elapsed, 'steps': count})
        arm['last_loss'] = float(loss.detach())
        if not math.isfinite(arm['last_loss']):
            raise FloatingPointError(f'{name}: nonfinite loss')
        return elapsed

    warm = config['warmup']
    for name in VARIANTS:
        block(name, warm, warmup=True)
    chunk = config['chunk']
    passes = 0
    recent_round_sec = None
    while min(a['steps'] for a in report['arms'].values()) < config['pilot_steps']:
        remaining = config['pilot_steps'] - min(a['steps'] for a in report['arms'].values())
        count = min(remaining, chunk)
        projected = recent_round_sec or sum(a['warmup_seconds'] for a in report['arms'].values()) * count/warm
        if budget.remaining_seconds() < projected * 1.4 + config['eval_reserve_seconds']:
            break
        order = VARIANTS[passes % 3:] + VARIANTS[:passes % 3]
        recent_round_sec = sum(block(name, count) for name in order)
        passes += 1
        if passes in (1, 2) or passes % 4 == 0:
            print('每组完成', report['arms']['plain']['steps'], '步；最近一轮三组耗时', round(recent_round_sec, 1), 's', flush=True)

    for name in VARIANTS:
        budget.check()
        sync()
        start_eval = time.perf_counter()
        arm = report['arms'][name]
        arm['validation'] = evaluate_loop(models[name], splits['val'], device, precision, config['batch'], budget)
        sync()
        arm['eval_seconds'] = time.perf_counter() - start_eval
    report['status'] = ('complete' if min(a['steps'] for a in report['arms'].values()) == config['pilot_steps']
                        else 'budget_shortened')


def show_and_save(report, output_dir):
    path = Path(output_dir) / 'sudoku_pilot_results.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False)
    path.write_text(payload, encoding='utf-8')
    print('\n试跑结果（条件翻转率按第 8 轮正确或错误的空格分别计算）')
    def pct(x):
        return '—' if x is None else f'{100*x:.2f}%'
    for name, arm in report.get('arms', {}).items():
        print(f"{NAMES[name]}：{arm['steps']} 步，{arm['params']:,} 个参数")
        for metric in arm.get('validation', []):
            print(f"  R{metric['depth']:>2}  空格 {pct(metric['blank_acc'])}  整盘 {pct(metric['board_acc'])}  "
                  f"对→错 {pct(metric.get('right_to_wrong'))}  错→对 {pct(metric.get('wrong_to_right'))}")
    f = report.get('forecast', {})
    print('\n同卡同费率，正式训练 3 组 × 3 个种子 × 5,000 步：')
    for name, row in f.get('by_model', {}).items():
        print(f"  {NAMES[name]}  {row['seconds_per_step']:.4f} 秒/步，计时 {row['timed_steps']} 步；3 个种子合计 {row['cu']:.2f} CU")
    if f.get('total'):
        total = f['total']
        print(f"预计 {total['hours']:.2f} GPU 小时 / {total['cu']:.2f} CU；较慢分段情景 {total['slower_cu']:.2f} CU；"
              f"基准加 20% 余量 {total['cu_plus_20pct']:.2f} CU。")
        if not total['includes_one_validation_per_seed']:
            print('部分验证计时缺失，上述数字仅包含已测量部分。')
    else:
        print('计时尚不完整，无法估算九次训练。')
    print('报告收录训练步数、分段原始秒数、翻转分母和 GPU 配置。', flush=True)
    display(HTML('<a download="sudoku_pilot_results.json" href="data:application/json;charset=utf-8,' + quote(payload) + '">下载本次试跑 JSON</a>'))
    # Full summary remains in saved cell output after Colab releases the VM.
    print('PILOT_SUMMARY ' + json.dumps({k: v for k, v in report.items() if k != 'arms'}, ensure_ascii=False), flush=True)


In [6]:
# @title 运行试验，显示正式训练预算，并释放此 Colab GPU
PREVIOUS_CU = 0.0  # @param {type:"number"}
CU_PER_HOUR = 0.0  # @param {type:"number"}
# 0 表示运行时手动输入当前面板费率。这里只会启动试跑。
CONFIG = dict(width=96, batch=64, train_loops=8, pilot_steps=512, warmup=16, chunk=16,
              formal_steps=5000, formal_seeds=3, n_train=2048, n_val=256, n_test=256,
              data_seed=2026, seed=17, batch_seed=1027, lr=0.001, eval_reserve_seconds=90)

def hosted_pilot():
    from google.colab import runtime
    import sys
    import threading
    from time import sleep
    report = {'status': 'starting'}
    budget = None
    rate = None
    timer = None
    output_dir = '/content/sudoku_pilot'
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('当前是 CPU。请选择 GPU 运行时后重新开始。')
        if CU_PER_HOUR == 0:
            timer = threading.Timer(180, release_colab_runtime)
            timer.daemon = True
            timer.start()
            rate = float(input('打开右上角「查看资源」，输入当前 GPU 的 CU/小时，再按回车：').strip())
        else:
            rate = float(CU_PER_HOUR)
        budget = RuntimeBudget(rate, PREVIOUS_CU, session_start=SUDOKU_SESSION_START)
        budget.arm_watchdog()
        if timer:
            timer.cancel()
        budget.check()
        run_pilot(budget, rate, report, CONFIG, output_dir)
    except BudgetStop as exc:
        report.update(status='budget_stopped', detail=str(exc))
    except (Exception, KeyboardInterrupt) as exc:
        import traceback
        traceback.print_exc()
        report.update(status='error', detail=f'{type(exc).__name__}: {exc}')
    finally:
        if timer:
            timer.cancel()
        try:
            if budget is not None:
                report['budget'] = budget.summary()
                report['forecast'] = formal_forecast(report.get('arms', {}), rate, CONFIG['formal_steps'],
                                                     CONFIG['formal_seeds'], CONFIG['warmup'])
            print('试跑状态', report['status'], report.get('detail', ''), flush=True)
            show_and_save(report, output_dir)
            print('报告已写入输出，准备释放 GPU。', flush=True)
            sys.stdout.flush()
            sleep(5)
        finally:
            release_colab_runtime()
            if budget:
                budget.disarm_watchdog()

hosted_pilot()


打开右上角「查看资源」，输入当前 GPU 的 CU/小时，再按回车：2.33
设备 NVIDIA L4，torch.bfloat16，试跑上限 10 CU。开始准备离线数独…
数据已就绪，用时 6.0s，验证题数 256。
每组完成 32 步；最近一轮三组耗时 1.0 s
每组完成 48 步；最近一轮三组耗时 1.0 s
每组完成 80 步；最近一轮三组耗时 1.1 s
每组完成 144 步；最近一轮三组耗时 1.0 s
每组完成 208 步；最近一轮三组耗时 1.0 s
每组完成 272 步；最近一轮三组耗时 1.0 s
每组完成 336 步；最近一轮三组耗时 1.0 s
每组完成 400 步；最近一轮三组耗时 1.0 s
每组完成 464 步；最近一轮三组耗时 1.0 s
试跑状态 complete 

试跑结果（条件翻转率按第 8 轮正确或错误的空格分别计算）
普通循环：512 步，79,401 个参数
  R 8  空格 93.11%  整盘 51.17%  对→错 —  错→对 —
  R16  空格 94.99%  整盘 68.36%  对→错 0.80%  错→对 38.13%
  R32  空格 94.82%  整盘 69.14%  对→错 1.09%  错→对 39.51%
全通道门控：512 步，82,697 个参数
  R 8  空格 93.55%  整盘 52.34%  对→错 —  错→对 —
  R16  空格 96.23%  整盘 75.00%  对→错 0.63%  错→对 50.66%
  R32  空格 96.62%  整盘 79.69%  对→错 0.69%  错→对 57.54%
分区门控：512 步，82,697 个参数
  R 8  空格 93.32%  整盘 50.39%  对→错 —  错→对 —
  R16  空格 95.15%  整盘 69.92%  对→错 0.94%  错→对 40.59%
  R32  空格 95.00%  整盘 71.48%  对→错 1.16%  错→对 41.44%

同卡同费率，正式训练 3 组 × 3 个种子 × 5,000 步：
  普通循环  0.0171 秒/步，计时 496 步；3 个种子合计 0.17 CU
  全通道门控  0.0238 秒/步，计时 496 步；3 个种

PILOT_SUMMARY {"status": "complete", "config": {"width": 96, "batch": 64, "train_loops": 8, "pilot_steps": 512, "warmup": 16, "chunk": 16, "formal_steps": 5000, "formal_seeds": 3, "n_train": 2048, "n_val": 256, "n_test": 256, "data_seed": 2026, "seed": 17, "batch_seed": 1027, "lr": 0.001, "eval_reserve_seconds": 90}, "gpu": "NVIDIA L4", "torch_version": "2.14.1+cu130", "precision": "torch.bfloat16", "cu_per_hour": 2.33, "data_seconds": 6.006201635999986, "budget": {"runtime_cu_per_hour_entered": 2.33, "previous_sessions_cu_entered": 0.0, "this_session_seconds": 74.88, "estimated_total_cu": 0.0485, "software_cu_limit": 8.0, "remaining_work_seconds": 6945.12}, "forecast": {"by_model": {"plain": {"seconds_per_step": 0.017106373814516095, "timed_steps": 496, "timing_blocks": 31, "hours": 0.0725493287821235, "cu": 0.16903993606234777, "slower_cu": 0.17356988331921733}, "full_gate": {"seconds_per_step": 0.023794009475806482, "timed_steps": 496, "timing_blocks": 31, "hours": 0.099278031710349